In [1]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import sklearn
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm

from stnd.utility.data_utils import make_or_load_from_cache
from stnd.utility.utils import apply_random_seed
import sys
import os
import sklearn
import numpy as np
import pandas as pd
from datasets import load_dataset
import json


FIRST_RUN = False
ROOT_PATH = os.path.join(os.path.dirname(os.path.abspath("")))
sys.path.insert(0, ROOT_PATH)
from experiments import (
    RANDOM_SEED,
    make_train_test_model_embeddings,
    make_cache_subpath,
    make_disagreement_scores_dict,
    make_fitted_weights
)
from utils import (
    lb_scenarios,
    dump_pickle,
    load_pickle,
    prepare_and_split_data
)
from plots import (
    MODEL_OUTPUTS_PATH,
    load_scores,
    safe_spearmanr
)
from selection import sample_items
from run_experiment import load_and_split_model_outputs
from acc import (
    compute_true_acc,
    compute_acc_knn
)
from scripts.download_leaderboard import MMLU_SUBSCENARIOS
# from utils_for_notebooks import read_per_model_info
from utils_for_notebooks import pad_predictions
from scripts.evaluate_mmlu import evaluate_mmlu
from scripts.extract_model_outputs_from_raw_data_v2 import MMLU_PRO_SCENARIO_SUFFIX
sys.path.pop(0);


CACHE_DIR = "./cache_dir"


/home/oh/arubinstein17/github/disco-public/envs/disco_env_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## functions

In [2]:
def _get_scenario_key(entry):
    """Return the key that ends with __leaderboard_mmlu_pro, or None."""
    if not isinstance(entry, dict):
        return None
    for k in entry:
        if k.endswith(MMLU_PRO_SCENARIO_SUFFIX):
            return k
    return None


def extract_source_outputs_from_v2_raw(raw_path, pad_to_size=31):
    """
    Load v2 leaderboard raw pickle and build source_outputs dict.

    Raw pickle format: {model_name: {scenario_name: {correctness, resps, ...}}}

    pad_to_size: size of the last dimension of predictions (pad with -inf).
    """
    raw = load_pickle(raw_path)

    # models = []
    scenario_key = None
    n_questions = None

    models_dict = {}

    for model_name, scenarios_dict in tqdm(raw.items()):
        if not isinstance(scenarios_dict, dict):
            continue
        sk = _get_scenario_key(scenarios_dict)
        if sk is None:
            continue
        rec = scenarios_dict[sk]
        if rec is None:
            continue
        correctness = rec.get("correctness")
        if correctness is None:
            continue
        try:
            nc = len(correctness)
        except TypeError:
            continue
        if scenario_key is None:
            scenario_key = sk
            n_questions = nc
        # elif sk != scenario_key or nc != n_questions:
        elif nc != n_questions:
            continue
        # models.append((model_name, rec))
        models_dict[model_name] = rec

    # if not models or scenario_key is None or n_questions is None:
    #     raise ValueError(
    #         "No valid model data found in raw pickle; need at least one model "
    #         "with non-None correctness for the MMLU-Pro scenario."
    #     )

    # n_models = len(models)
    # n_answers = 1

    # correctness_arr = np.zeros((n_models, n_questions, 1), dtype=np.float64)
    # predictions_arr = np.full(
    #     (n_models, n_questions, pad_to_size), -np.inf, dtype=np.float64
    # )

    return models_dict

    # for i, (model_name, rec) in enumerate(models):
    #     corr = rec["correctness"]
    #     if hasattr(corr, "__iter__") and not isinstance(corr, (str, bytes)):
    #         corr = list(corr)
    #     else:
    #         corr = [float(corr)] * n_questions
    #     correctness_arr[i, :, 0] = np.array(
    #         corr[:n_questions], dtype=np.float64
    #     )

    #     resps = rec.get("resps") or rec.get("filtered_resps")
    #     pred = _resps_to_prediction_indices(resps, n_questions, max_answers=1)
    #     predictions_arr[i, :, :1] = pred[:, :1]

    # # Same format as two_stages.build_outputs_dict / source_outputs.pkl
    # scenarios_map = {scenario_key: list(range(n_questions))}
    # models_map = {name: i for i, (name, _) in enumerate(models)}
    # datapoints_map = {i: i for i in range(n_questions)}

    # return {
    #     "predictions": predictions_arr,
    #     "correctness": correctness_arr,
    #     "Models": models_map,
    #     "Datapoints": datapoints_map,
    #     "Scenarios": scenarios_map,
    # }

## check remote responses

### Save single model

In [4]:
# model_name = 'open-llm-leaderboard/JungZoona__T3Q-qwen2.5-14b-v1.0-e3-details'
model_name = "open-llm-leaderboard/01-ai__Yi-9B-details"

In [16]:
if FIRST_RUN:
    raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw_all.pickle"
    # raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw.pickle"
    models_dict = extract_source_outputs_from_v2_raw(raw_path)
# 12 min loading

100%|██████████| 6505/6505 [00:00<00:00, 874457.47it/s]


In [5]:
import pickle

# model_name = 'open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'

if FIRST_RUN:
    with open(model_name.replace("/", "_") + ".pkl", "wb") as f:
        pickle.dump(models_dict[model_name], f)

In [6]:
with open(model_name.replace("/", "_") + ".pkl", "rb") as f:
    single_model_dict = pickle.load(f)

In [9]:
print(single_model_dict.keys())

dict_keys(['dates', 'doc', 'resps', 'filtered_resps', 'target', 'arguments', 'correctness'])


In [8]:
single_model_dict['resps']

Column([[[['-3.1902904510498047', 'False']], [['-3.4402904510498047', 'False']], [['-3.8152904510498047', 'False']], [['-2.6902904510498047', 'False']], [['-1.0652905702590942', 'True']], [['-2.1902904510498047', 'False']], [['-3.9402904510498047', 'False']], [['-4.690290451049805', 'False']], [['-1.1902905702590942', 'False']]], [[['-4.589632034301758', 'False']], [['-5.339632034301758', 'False']], [['-5.964632034301758', 'False']], [['-5.964632034301758', 'False']], [['-0.08963188529014587', 'True']], [['-7.339632034301758', 'False']], [['-7.214632034301758', 'False']], [['-8.089632034301758', 'False']], [['-7.839632034301758', 'False']], [['-5.839632034301758', 'False']]], [[['-1.7078464031219482', 'False']], [['-2.0828464031219482', 'False']], [['-2.3328464031219482', 'False']], [['-1.5828464031219482', 'True']], [['-1.9578464031219482', 'False']], [['-2.9578464031219482', 'False']], [['-3.2078464031219482', 'False']], [['-4.082846641540527', 'False']], [['-3.7078464031219482', 'Fa

In [14]:
import pandas as pd
import random

csv_path = "/home/oh/arubinstein17/github/disco-public/benchmark_csvs/open-llm-leaderboard-v2.csv"
df = pd.read_csv(csv_path)

filtered_df = df[df['Type'].str.contains("pretrained", na=False)]
filtered_df = df[df['fullname'].str.contains("/", na=False)]
SELECTED_MODELS = random.sample(list(filtered_df['fullname']), 10)
print(SELECTED_MODELS)

['Deci/DeciLM-7B-instruct', 'Sakalti/SJT-8B-V1.1', 'T145/ZEUS-8B-V26', 'marcuscedricridia/Hush-Qwen2.5-7B-v1.2', 'godlikehhd/alpaca_data_score_max_0.7_2600', '3rd-Degree-Burn/L-3.1-Science-Writer-8B', 'ClaudioItaly/Book-Gut12B', 'jaspionjader/bh-19', 'Lawnakk/BBALAW1.61', 'fblgit/miniclaus-qw1.5B-UNAMGS']


In [19]:
if FIRST_RUN:
    dict_to_save = {}
    for model_name in SELECTED_MODELS:
        dict_key = "open-llm-leaderboard/" + model_name.replace("/", "__") + "-details"
        dict_to_save[model_name] = models_dict[dict_key]
    with open("selected_models.pkl", "wb") as f:
        pickle.dump(dict_to_save, f)

In [3]:
with open("selected_models.pkl", "rb") as f:
    candidate_models = pickle.load(f)

In [4]:
print(candidate_models.keys())

dict_keys(['Deci/DeciLM-7B-instruct', 'Sakalti/SJT-8B-V1.1', 'T145/ZEUS-8B-V26', 'marcuscedricridia/Hush-Qwen2.5-7B-v1.2', 'godlikehhd/alpaca_data_score_max_0.7_2600', '3rd-Degree-Burn/L-3.1-Science-Writer-8B', 'ClaudioItaly/Book-Gut12B', 'jaspionjader/bh-19', 'Lawnakk/BBALAW1.61', 'fblgit/miniclaus-qw1.5B-UNAMGS'])


### Debug

In [7]:

model_name_lucie = "open-llm-leaderboard/OpenLLM-France__Lucie-7B-details"

In [18]:
import pickle

# model_name = 'open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'

if FIRST_RUN:
    with open(model_name.replace("/", "_") + ".pkl", "wb") as f:
        pickle.dump(models_dict[model_name], f)

In [8]:
with open(model_name_lucie.replace("/", "_") + ".pkl", "rb") as f:
    single_model_dict_lucie = pickle.load(f)

In [20]:
# raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw_all.pickle"
raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw.pickle"
models_dict = extract_source_outputs_from_v2_raw(raw_path)

100%|██████████| 4/4 [00:00<00:00, 73908.44it/s]


In [21]:
models_dict.keys()

dict_keys(['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details', 'open-llm-leaderboard/LeroyDyer__SpydazWeb_AI_HumanAI_011_INSTRUCT-details', 'open-llm-leaderboard/Youlln__ECE-MIRAGE-1-12B-details', 'open-llm-leaderboard/jaspionjader__bh-39-details'])

In [ ]:
models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']

In [23]:
models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'].keys()

dict_keys(['dates', 'doc', 'resps', 'filtered_resps', 'target', 'arguments', 'correctness'])

In [24]:
import pickle

with open("matter_0.2-7B-DPO-details.pkl", "wb") as f:
    pickle.dump(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'], f)



In [25]:
with open("matter_0.2-7B-DPO-details.pkl", "rb") as f:
    single_model_dict = pickle.load(f)

In [26]:
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["arguments"]))
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["filtered_resps"]))
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["resps"]))
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["correctness"]))




12032
12032
12032
12032


## Save mmlu_prompts from local responses

In [12]:
local_path = "/home/oh/arubinstein17/github/lm-eval-fork/output/mmlu_pro/llmeval135/01-ai__Yi-9B/samples_leaderboard_mmlu_pro_2026-02-14T11-47-10.579194.jsonl"



In [ ]:
# Extract mmlu_pro_prompts_examples from gen_args_0.arg_0 and save in mmlu_prompts_examples.json format
import string

def doc_to_query(doc):
    """Build query string from doc (question + options + Answer:)."""
    s = f"{doc['question']}\n"
    for i in range(len(doc["options"])):
        s += f"{string.ascii_uppercase[i]}. {doc['options'][i]}\n"
    s += "Answer:"
    return s

def extract_mmlu_pro_prompts_examples(jsonl_path, output_path):
    """Extract prompts in mmlu_prompts_examples format from lm-eval JSONL samples."""
    examples = []
    with open(jsonl_path) as f:
        for line in f:
            rec = json.loads(line)
            doc = rec["doc"]
            arg_0 = rec["arguments"]["gen_args_0"]["arg_0"]  # full_prompt
            query = doc_to_query(doc)
            choices = [string.ascii_uppercase[i] for i in range(len(doc["options"]))]
            gold = doc["answer_index"]
            example_str = query + " " + doc["answer"]
            examples.append({
                "example": example_str,
                "full_prompt": arg_0,
                "query": query,
                "choices": choices,
                "gold": gold,
            })
    with open(output_path, "w") as out:
        json.dump(examples, out, indent=2)
    print(f"Saved {len(examples)} records to {output_path}")
    return examples

out_path = os.path.join(ROOT_PATH, "notebooks", "mmlu_pro_prompts_examples.json")
extract_mmlu_pro_prompts_examples(local_path, out_path)

## Save mmlu_prompts examples from remote data

In [5]:
# Save full_prompts for the first model from candidate_models in mmlu_prompts_examples format
import string

def doc_to_query(doc):
    """Build query string from doc (question + options + Answer:)."""
    s = f"{doc['question']}\n"
    for i in range(len(doc["options"])):
        s += f"{string.ascii_uppercase[i]}. {doc['options'][i]}\n"
    s += "Answer:"
    return s

first_model_name = next(iter(candidate_models.keys()))
d = candidate_models[first_model_name]
examples = []
for i in range(len(d["doc"])):
    doc = d["doc"][i]
    arg_0 = d["arguments"][i]["gen_args_0"]["arg_0"]
    query = doc_to_query(doc)
    choices = [string.ascii_uppercase[j] for j in range(len(doc["options"]))]
    gold = doc["answer_index"]
    example_str = query + " " + doc["answer"]
    examples.append({
        "example": example_str,
        "full_prompt": arg_0,
        "query": query,
        "choices": choices,
        "gold": gold,
    })

out_path = os.path.join(ROOT_PATH, "notebooks", "mmlu_pro_prompts_examples_first_model.json")
with open(out_path, "w") as out:
    json.dump(examples, out, indent=2)
print(f"Saved {len(examples)} records for first model {first_model_name!r} to {out_path}")

Saved 12032 records for first model 'Deci/DeciLM-7B-instruct' to /weka/oh/arubinstein17/github/disco-public/notebooks/mmlu_pro_prompts_examples_first_model.json


## Compare local vs remote saved questions

In [8]:
# Compare local eval prompts vs first-model (remote) prompts using same rules as the 10-model check
import string

def _normalize_query(q):
    """Normalize query for matching between datasets."""
    return q.strip().replace("Answer:", "").strip()

local_path = "/home/oh/arubinstein17/github/disco-public/notebooks/mmlu_pro_prompts_examples.json"
first_model_path = "/home/oh/arubinstein17/github/disco-public/notebooks/mmlu_pro_prompts_examples_first_model.json"

with open(local_path) as f:
    local_examples = json.load(f)
with open(first_model_path) as f:
    first_model_examples = json.load(f)

local_by_query = {_normalize_query(ex["query"]): ex for ex in local_examples}
first_by_query = {_normalize_query(ex["query"]): ex for ex in first_model_examples}
common_queries = set(local_by_query) & set(first_by_query)
only_local = set(local_by_query) - set(first_by_query)
only_first = set(first_by_query) - set(local_by_query)

N_EXAMPLES = 5
n_allowed_mismatches = 100

n_ends_with_fail_local = 0
n_ends_with_fail_first = 0
first_fail_local = None
first_fail_first = None

for q_norm in common_queries:
    local_ex = local_by_query[q_norm]
    first_ex = first_by_query[q_norm]
    ref_query_stripped = q_norm
    fp_local = " ".join(local_ex["full_prompt"].split("Answer:")[:-1]).replace("Answer:", "")
    if not (fp_local.strip().endswith(ref_query_stripped) or ref_query_stripped in fp_local[-len(ref_query_stripped)-50:]):
        n_ends_with_fail_local += 1
        if first_fail_local is None:
            first_fail_local = (q_norm[:80], fp_local[-100:] if len(fp_local) > 100 else fp_local)
    fp_first = " ".join(first_ex["full_prompt"].split("Answer:")[:-1]).replace("Answer:", "")
    if not (fp_first.strip().endswith(ref_query_stripped) or ref_query_stripped in fp_first[-len(ref_query_stripped)-50:]):
        n_ends_with_fail_first += 1
        if first_fail_first is None:
            first_fail_first = (q_norm[:80], fp_first[-100:] if len(fp_first) > 100 else fp_first)

ends_with_ok_local = n_ends_with_fail_local <= n_allowed_mismatches
ends_with_ok_first = n_ends_with_fail_first <= n_allowed_mismatches
n_answer_fail_local = sum(1 for q in common_queries if local_by_query[q]["full_prompt"].count("Answer:") < N_EXAMPLES + 1)
n_answer_fail_first = sum(1 for q in common_queries if first_by_query[q]["full_prompt"].count("Answer:") < N_EXAMPLES + 1)
answer_count_ok_local = n_answer_fail_local <= 10
answer_count_ok_first = n_answer_fail_first <= 10

queries_diff = {
    "only_in_local": [local_by_query[q]["query"] for q in only_local],
    "only_in_first_model": [first_by_query[q]["query"] for q in only_first],
    "n_common": len(common_queries),
    "n_only_local": len(only_local),
    "n_only_first_model": len(only_first),
}
queries_diff_path = os.path.join(ROOT_PATH, "notebooks", "queries_diff.json")
with open(queries_diff_path, "w") as f:
    json.dump(queries_diff, f, indent=2)
print(f"Saved query diff to {queries_diff_path}")

print("--- Compare local vs first_model saved prompts ---")
print(f"Local: {len(local_examples)} examples. First model: {len(first_model_examples)} examples.")
print(f"Common queries: {len(common_queries)}. Only in local: {len(only_local)}. Only in first_model: {len(only_first)}.")
print()
print("(1) full_prompt ends with the corresponding query:")
print(f"  Local:       {n_ends_with_fail_local} mismatches (ok={ends_with_ok_local})")
print(f"  First model: {n_ends_with_fail_first} mismatches (ok={ends_with_ok_first})")
if first_fail_local:
    print(f"  First local failure: query_prefix={first_fail_local[0]!r}...")
if first_fail_first:
    print(f"  First first_model failure: query_prefix={first_fail_first[0]!r}...")
print()
print(f"(2) full_prompt has at least {N_EXAMPLES+1} 'Answer:' (5 examples + query):")
print(f"  Local:       {n_answer_fail_local} fail (ok={answer_count_ok_local})")
print(f"  First model: {n_answer_fail_first} fail (ok={answer_count_ok_first})")

Saved query diff to /weka/oh/arubinstein17/github/disco-public/notebooks/queries_diff.json
--- Compare local vs first_model saved prompts ---
Local: 12032 examples. First model: 12032 examples.
Common queries: 11381. Only in local: 651. Only in first_model: 492.

(1) full_prompt ends with the corresponding query:
  Local:       0 mismatches (ok=True)
  First model: 0 mismatches (ok=True)

(2) full_prompt has at least 6 'Answer:' (5 examples + query):
  Local:       0 fail (ok=True)
  First model: 0 fail (ok=True)


## Compare local vs remote questions

In [16]:
# For all models in candidate_models, find the maximum number_queries such that the first number_queries are identical across all models (with optional allowed consecutive mismatches).
# Parametrized by which field to compare: "query" (from doc) or "full_prompt" (from arguments[i]["gen_args_0"]["arg_0"]).
import string

def doc_to_query(doc):
    """Build query string from doc (question + options + Answer:)."""
    s = f"{doc['question']}\n"
    for i in range(len(doc["options"])):
        s += f"{string.ascii_uppercase[i]}. {doc['options'][i]}\n"
    s += "Answer:"
    return s

def get_value_at_index(d, i, field):
    """Return the value at index i for comparison. field in ('query', 'full_prompt')."""
    if field == "query":
        return doc_to_query(d["doc"][i])
    elif field == "full_prompt":
        return d["arguments"][i]["gen_args_0"]["arg_0"]
    else:
        raise ValueError(f"field must be 'query' or 'full_prompt', got {field!r}")

def max_identical_prefix(candidate_models, field, allow_n_mismatches=0):
    """
    Find the largest N such that the first N values (queries or full_prompts) are identical across all models,
    allowing at most allow_n_mismatches consecutive mismatches.
    Returns (number_queries, model_value_lists, names, min_len).
    """
    model_value_lists = {}
    for name, d in candidate_models.items():
        n = len(d["doc"])
        model_value_lists[name] = [get_value_at_index(d, i, field) for i in range(n)]
    min_len = min(len(v) for v in model_value_lists.values())
    names = list(model_value_lists.keys())

    number_queries = 0
    n_mismatches = 0
    for i in range(min_len):
        ref = model_value_lists[names[0]][i]
        if all(model_value_lists[n][i] == ref for n in names):
            number_queries += 1
            # consecutive_mismatches = 0
        else:
            if allow_n_mismatches == 0:
                break
            # if consecutive_mismatches < allow_n_mismatches:
            if n_mismatches < allow_n_mismatches:
                number_queries += 1
                # consecutive_mismatches += 1
                n_mismatches += 1
            else:
                break
    return number_queries, model_value_lists, names, min_len

allow_n_mismatches = 10  # set > 0 to allow that many consecutive mismatches

# Run for queries (from doc)
number_queries_q, model_query_lists, names, min_len = max_identical_prefix(candidate_models, "query", allow_n_mismatches)
print(f"--- field = 'query' ---")
print(f"Min length across models: {min_len}")
print(f"allow_n_mismatches: {allow_n_mismatches}")
print(f"Max prefix length with identical queries: {number_queries_q}")
if number_queries_q < min_len:
    ref = model_query_lists[names[0]][number_queries_q]
    diffs = [n for n in names if model_query_lists[n][number_queries_q] != ref]
    print(f"First differing index: {number_queries_q} (models with different query: {diffs[:5]}...)")

# Run for full_prompt (from arguments)
number_queries_fp, model_fp_lists, _, _ = max_identical_prefix(candidate_models, "full_prompt", allow_n_mismatches)
print(f"\n--- field = 'full_prompt' ---")
print(f"Max prefix length with identical full_prompts: {number_queries_fp}")
if number_queries_fp < min_len:
    ref = model_fp_lists[names[0]][number_queries_fp]
    diffs = [n for n in names if model_fp_lists[n][number_queries_fp] != ref]
    print(f"First differing index: {number_queries_fp} (models with different full_prompt: {diffs[:5]}...)")

--- field = 'query' ---
Min length across models: 12032
allow_n_mismatches: 10
Max prefix length with identical queries: 12032

--- field = 'full_prompt' ---
Max prefix length with identical full_prompts: 10
First differing index: 10 (models with different full_prompt: ['Sakalti/SJT-8B-V1.1', 'T145/ZEUS-8B-V26', 'marcuscedricridia/Hush-Qwen2.5-7B-v1.2', 'godlikehhd/alpaca_data_score_max_0.7_2600', '3rd-Degree-Burn/L-3.1-Science-Writer-8B']...)


In [7]:
# Check that for each model full_prompt = [5 examples (can differ per model)] + [same query for all models at that index]
import string

def doc_to_query(doc):
    """Build query string from doc (question + options + Answer:)."""
    s = f"{doc['question']}\n"
    for i in range(len(doc["options"])):
        s += f"{string.ascii_uppercase[i]}. {doc['options'][i]}\n"
    s += "Answer:"
    return s

N_EXAMPLES = 5  # expected number of few-shot examples before the query
n_allowed_mismatches = 100  # at most this many (model, index) failures allowed for each check
min_len = min(len(d["doc"]) for d in candidate_models.values())
names = list(candidate_models.keys())

# (1) At each index i, every model's full_prompt should end with the same query (the one from doc i).
n_ends_with_fail = 0
first_fail = None  # (model_name, index)
for i in range(min_len):
    ref_query = doc_to_query(list(candidate_models.values())[0]["doc"][i])
    ref_query_stripped = ref_query.strip()
    ref_query_stripped = ref_query_stripped.replace("Answer:", "") # some models do not have "Answer:" in the query
    for name, d in candidate_models.items():
        # fp = d["arguments"][i]["gen_args_0"]["arg_0"]
        fp = " ".join(d["arguments"][i]["gen_args_0"]["arg_0"].split("Answer:")[:-1])
        fp = fp.replace("Answer:", "") # some models do not have "Answer:" in the query

        if not (fp.strip().endswith(ref_query_stripped) or ref_query_stripped in fp[-len(ref_query_stripped)-50:]):
            if first_fail is None:
                print(f"First failure: model={name!r}, index={i}")
                print(f"ref_query: {ref_query_stripped}")
                print(f"fp: {fp}")
            n_ends_with_fail += 1
            if first_fail is None:
                first_fail = (name, i)
ends_with_query_ok = n_ends_with_fail <= n_allowed_mismatches

print("--- full_prompt structure: [examples] + [same query] ---")
print(f"n_allowed_mismatches: {n_allowed_mismatches}")
print(f"full_prompts end with the corresponding query: {n_ends_with_fail} mismatches (ok={ends_with_query_ok})")
if first_fail:
    print(f"First failure: model={first_fail[0]!r}, index={first_fail[1]}")

# (2) full_prompt should contain "Answer:" at least N_EXAMPLES+1 times (5 examples + query), sanity check for structure.

n_allowed_mismatches = 10
n_answer_ok = 0
n_answer_fail = 0
first_prefix_fail = None
for i in range(min_len):
    for name, d in candidate_models.items():
        fp = d["arguments"][i]["gen_args_0"]["arg_0"]
        # fp = " ".join(d["arguments"][i]["gen_args_0"]["arg_0"].split("Answer:")[:-1])
        count = fp.count("Answer:")
        if count >= N_EXAMPLES + 1:
            n_answer_ok += 1
        else:
            n_answer_fail += 1
            if first_prefix_fail is None:
                first_prefix_fail = (name, i, count)
total_checks = min_len * len(candidate_models)
answer_count_ok = n_answer_fail <= n_allowed_mismatches
print(f"full_prompt has at least {N_EXAMPLES+1} 'Answer:' (5 examples + query): {n_answer_ok}/{total_checks} pass, {n_answer_fail} fail (ok={answer_count_ok})")
if first_prefix_fail:
    print(f"First fail: model={first_prefix_fail[0]!r}, index={first_prefix_fail[1]}, count(Answer:)={first_prefix_fail[2]}")

First failure: model='Sakalti/SJT-8B-V1.1', index=1535
ref_query: Suppose there are 100 identical firms in a perfectly competitive industry. Each firm has a short-run total cost function of the form C(q) = rac{1}{300}q^3 + 0.2q^2 + 4q + 10. Suppose market demand is given by Q = -200P + 8,000. What will be the short-run equilibrium price?
A. 25
B. 28
C. 30
D. 50
E. 20
F. 15
G. 35
H. 45
I. 10
J. 40

fp: The symmetric group $S_n$ has $
\factorial{n}$ elements, hence it is not true that $S_{10}$ has 10 elements.
Find the characteristic of the ring 2Z.
A. 0
B. 30
C. 3
D. 10
E. 12
F. 50
G. 2
H. 100
I. 20
J. 5
  A

Let V be the set of all real polynomials p(x). Let transformations T, S be defined on V by T:p(x) -> xp(x) and S:p(x) -> p'(x) = d/dx p(x), and interpret (ST)(p(x)) as S(T(p(x))). Which of the following is true?
A. ST + TS is the identity map of V onto itself.
B. TS = 0
C. ST = 1
D. ST - TS = 0
E. ST = T
F. ST = 0
G. ST = TS
H. ST - TS is the identity map of V onto itself.
I. TS =

In [ ]:
# single_model_dict = single_model_dict_lucie

In [21]:
# Compare questions: mmlu_pro_prompts_examples.json (from local JSONL) vs single_model_dict (from leaderboard pickle)
import string

def doc_to_query(doc):
    """Build query string from doc (question + options + Answer:)."""
    s = f"{doc['question']}\n"
    for i in range(len(doc["options"])):
        s += f"{string.ascii_uppercase[i]}. {doc['options'][i]}\n"
    s += "Answer:"
    return s

out_path = os.path.join(ROOT_PATH, "notebooks", "mmlu_pro_prompts_examples.json")
with open(out_path) as f:
    json_examples = json.load(f)

n_json = len(json_examples)
n_pickle = len(single_model_dict["doc"])
print(f"Count: JSON={n_json}, pickle={n_pickle}, match={n_json == n_pickle}")

if n_json != n_pickle:
    print("Length mismatch — cannot compare element-wise.")
else:
    query_matches = []
    full_prompt_matches = []
    for i in range(n_json):
        json_query = json_examples[i]["query"]
        json_full_prompt = json_examples[i]["full_prompt"]
        doc_i = single_model_dict["doc"][i]
        pickle_query = doc_to_query(doc_i)
        pickle_full_prompt = single_model_dict["arguments"][i]["gen_args_0"]["arg_0"]
        query_matches.append(json_query == pickle_query)
        full_prompt_matches.append(json_full_prompt == pickle_full_prompt)
    print(f"Query match: {sum(query_matches)}/{n_json}")
    print(f"Full prompt match: {sum(full_prompt_matches)}/{n_json}")
    if sum(query_matches) != n_json or sum(full_prompt_matches) != n_json:
        diff_idxs = [i for i in range(n_json) if not query_matches[i] or not full_prompt_matches[i]]
        print(f"First few differing indices: {diff_idxs[:10]}")

# Check whether queries match up to permutation (same set/multiset of queries)
from collections import Counter

json_queries = [ex["query"] for ex in json_examples]
pickle_queries = [doc_to_query(doc) for doc in single_model_dict["doc"]]

json_set = set(json_queries)
pickle_set = set(pickle_queries)
json_in_pickle = json_set <= pickle_set
pickle_in_json = pickle_set <= json_set
print(f"\n--- Match up to permutation (set of queries) ---")
print(f"Every JSON query appears in pickle: {json_in_pickle} (JSON set size {len(json_set)}, pickle set size {len(pickle_set)})")
print(f"Every pickle query appears in JSON: {pickle_in_json}")
if not json_in_pickle:
    only_in_json = json_set - pickle_set
    print(f"Queries in JSON but not in pickle: {len(only_in_json)} (first 3: {list(only_in_json)[:3]})")
if not pickle_in_json:
    only_in_pickle = pickle_set - json_set
    print(f"Queries in pickle but not in JSON: {len(only_in_pickle)} (first 3: {list(only_in_pickle)[:3]})")

json_counts = Counter(json_queries)
pickle_counts = Counter(pickle_queries)
multiset_match = json_counts == pickle_counts
print(f"Same multiset (identical counts per query): {multiset_match}")

Count: JSON=12032, pickle=12032, match=True
Query match: 2/12032
Full prompt match: 2/12032
First few differing indices: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

--- Match up to permutation (set of queries) ---
Every JSON query appears in pickle: False (JSON set size 12032, pickle set size 11874)
Every pickle query appears in JSON: False
Queries in JSON but not in pickle: 644 (first 3: ['25. Consider the initial value problem\n$$\n2 y^{\\prime \\prime}+3 y^{\\prime}-2 y=0, \\quad y(0)=1, \\quad y^{\\prime}(0)=-\\beta,\n$$\nwhere $\\beta>0$.\nFind the smallest value of $\\beta$ for which the solution has no minimum point.\nA. 7\nB. 2.5\nC. 4\nD. 1\nE. 2\nF. 6\nG. 5\nH. 1.5\nI. 0.5\nJ. 3\nAnswer:', 'A particle of mass $m$ and velocity $u_1$ makes a head-on collision with another particle of mass $2 m$ at rest. If the coefficient of restitution is such to make the loss of total kinetic energy a maximum, what are the velocities $v_1$ after the collision?\nA. $0$\nB. $\\frac{1}{3}$$u_1$\nC. $\\frac

In [9]:
# Compare single_model_dict_lucie vs single_model_dict: identical queries (and full_prompt structure: 5 examples + query)
import string
from collections import Counter

def doc_to_query(doc):
    """Build query string from doc (question + options + Answer:)."""
    s = f"{doc['question']}\n"
    for i in range(len(doc["options"])):
        s += f"{string.ascii_uppercase[i]}. {doc['options'][i]}\n"
    s += "Answer:"
    return s

lucie_queries = [doc_to_query(doc) for doc in single_model_dict_lucie["doc"]]
other_queries = [doc_to_query(doc) for doc in single_model_dict["doc"]]

n_lucie = len(lucie_queries)
n_other = len(other_queries)
print(f"Count: lucie={n_lucie}, single_model_dict={n_other}, match={n_lucie == n_other}")

# Set comparison: same set of queries?
lucie_set = set(lucie_queries)
other_set = set(other_queries)
lucie_in_other = lucie_set <= other_set
other_in_lucie = other_set <= lucie_set
print(f"\n--- Match up to permutation (set of queries) ---")
print(f"Every lucie query appears in single_model_dict: {lucie_in_other} (lucie set size {len(lucie_set)}, other set size {len(other_set)})")
print(f"Every single_model_dict query appears in lucie: {other_in_lucie}")
if not lucie_in_other:
    only_lucie = lucie_set - other_set
    print(f"Queries in lucie but not in single_model_dict: {len(only_lucie)} (first 3: {list(only_lucie)[:3]})")
if not other_in_lucie:
    only_other = other_set - lucie_set
    print(f"Queries in single_model_dict but not in lucie: {len(only_other)} (first 3: {list(only_other)[:3]})")

# Multiset: same counts per query?
lucie_counts = Counter(lucie_queries)
other_counts = Counter(other_queries)
multiset_match = lucie_counts == other_counts
print(f"Same multiset (identical counts per query): {multiset_match}")

# Structure check: full_prompt ends with the corresponding query (5 examples + query)
def full_prompt_ends_with_query(d, idx):
    full_prompt = d["arguments"][idx]["gen_args_0"]["arg_0"]
    query = doc_to_query(d["doc"][idx])
    return full_prompt.strip().endswith(query.strip()) or query.strip() in full_prompt[-len(query)-50:]

lucie_structure_ok = all(full_prompt_ends_with_query(single_model_dict_lucie, i) for i in range(n_lucie))
other_structure_ok = all(full_prompt_ends_with_query(single_model_dict, i) for i in range(n_other))
print(f"\n--- Full prompt structure (ends with query) ---")
print(f"Lucie: all full_prompts end with corresponding query: {lucie_structure_ok}")
print(f"single_model_dict: all full_prompts end with corresponding query: {other_structure_ok}")

Count: lucie=12032, single_model_dict=12032, match=True

--- Match up to permutation (set of queries) ---
Every lucie query appears in single_model_dict: False (lucie set size 11874, other set size 11873)
Every single_model_dict query appears in lucie: False
Queries in lucie but not in single_model_dict: 7 (first 3: ['Let {N(t), t=[0, \\infty]} be a Poisson process with rate $\\lambda = 0.5$. Find the probability of no arrivals in [3, 5)\nA. 0.82\nB. 0.99\nC. 0.50\nD. 0.75\nE. 0.25\nF. 0.01\nG. 0.67\nH. 0.37\nI. 0.60\nJ. 0.55\nAnswer:', 'Which of the following is a renewable energy source?\nA. Oil shale\nB. Wood\nC. Natural gas\nD. Oil\nE. Peat\nF. Diesel\nG. Gasoline\nH. Propane\nI. Coal\nJ. Uranium\nAnswer:', 'Suppose there are 100 identical firms in a perfectly competitive industry. Each firm has a short-run total cost function of the form C(q) = \\frac{1}{300}q^3 + 0.2q^2 + 4q + 10. Suppose market demand is given by Q = -200P + 8,000. What will be the short-run equilibrium price?\n